# Module 7 — Exercises: AIOps Pipeline
**Nutanix AI/ML Intermediate Workshop**

These exercises reinforce every section of the Module 7 capstone.
Work through them in order — each builds on the previous.

> **Setup:** Run the capstone notebook (Sections 1–7) before starting these exercises.
> All models, helpers, and imports from the capstone are assumed to be in memory.

---
## Exercise 1 — Telemetry Generation (Section 1)
**Difficulty:** ⭐ Beginner

The capstone generates 1000 normal CVM samples. Your task: generate a **custom scenario**.

### Task
Write a function `generate_scenario(scenario: str, n: int) -> pd.DataFrame` that returns
`n` rows of telemetry for one of three scenarios:

| Scenario | CPU | Memory | IOPS | Latency |
|----------|-----|--------|------|---------|
| `"normal"` | 20–50% | 40–65% | 5000–12000 | 1–5 ms |
| `"cpu_spike"` | 85–99% | 75–95% | 15000–25000 | 30–80 ms |
| `"network_storm"` | 50–70% | 50–70% | 6000–10000 | 20–50 ms |

Each row must also have: `node` (randomly one of node-1 to node-4),
`disk_read_mbps`, `disk_write_mbps`, `network_mbps`, `stargate_ops_per_sec`.

**Acceptance test:** `generate_scenario("cpu_spike", 50)` returns a DataFrame with
50 rows where `cpu_pct.mean() > 85`.

In [ ]:
import numpy as np
import pandas as pd

# ── YOUR CODE HERE ────────────────────────────────────────────────────────
def generate_scenario(scenario: str, n: int) -> pd.DataFrame:
    np.random.seed(42)
    nodes = [f'node-{i}' for i in range(1, 5)]
    
    # Define ranges per scenario
    ranges = {
        'normal':        dict(cpu=(20,50),   mem=(40,65),  iops=(5000,12000),  lat=(1,5),    net=(200,800),   sops=(1500,2500)),
        'cpu_spike':     dict(cpu=(85,99),   mem=(75,95),  iops=(15000,25000), lat=(30,80),  net=(400,900),   sops=(200,600)),
        'network_storm': dict(cpu=(50,70),   mem=(50,70),  iops=(6000,10000),  lat=(20,50),  net=(1500,2500), sops=(1200,2000)),
    }
    
    r = ranges[scenario]
    
    def rnd(lo, hi): return np.random.uniform(lo, hi, n)
    
    return pd.DataFrame({
        'node':                 np.random.choice(nodes, n),
        'cpu_pct':              rnd(*r['cpu']),
        'mem_pct':              rnd(*r['mem']),
        'iops':                 rnd(*r['iops']),
        'latency_ms':           rnd(*r['lat']),
        'disk_read_mbps':       rnd(80, 200),
        'disk_write_mbps':      rnd(50, 150),
        'network_mbps':         rnd(*r['net']),
        'stargate_ops_per_sec': rnd(*r['sops']),
    })


# ── Acceptance test ───────────────────────────────────────────────────────
df_spike = generate_scenario('cpu_spike', 50)
assert df_spike.shape[0] == 50, "Must return 50 rows"
assert df_spike['cpu_pct'].mean() > 85, f"cpu_pct mean should be >85, got {df_spike['cpu_pct'].mean():.1f}"
print("Exercise 1 passed ✅")
print(df_spike[['node','cpu_pct','mem_pct','iops','latency_ms']].describe().round(1))

---
## Exercise 2 — Feature Engineering (Section 2)
**Difficulty:** ⭐⭐ Intermediate

The capstone adds rolling stats and derived ratios. Your task: add **two new features**.

### Task
Extend the `engineer_features` function to also compute:

1. **`disk_pressure`** = `(disk_read_mbps + disk_write_mbps) / stargate_ops_per_sec`
   — measures disk throughput per Stargate operation (higher = more disk stress per op)

2. **`network_cpu_ratio`** = `network_mbps / (cpu_pct + 1)`
   — measures network load relative to CPU (helps detect network-induced CPU stalls)

**Acceptance test:** Both columns exist and contain no NaN values.

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────
def add_custom_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['disk_pressure']      = (df['disk_read_mbps'] + df['disk_write_mbps']) / (df['stargate_ops_per_sec'] + 0.001)
    df['network_cpu_ratio']  = df['network_mbps'] / (df['cpu_pct'] + 1)
    return df


# ── Acceptance test ───────────────────────────────────────────────────────
df_normal = generate_scenario('normal', 100)
df_feat   = add_custom_features(df_normal)

assert 'disk_pressure'     in df_feat.columns, "disk_pressure column missing"
assert 'network_cpu_ratio' in df_feat.columns, "network_cpu_ratio column missing"
assert df_feat['disk_pressure'].isna().sum() == 0, "disk_pressure has NaN"
assert df_feat['network_cpu_ratio'].isna().sum() == 0, "network_cpu_ratio has NaN"

print("Exercise 2 passed ✅")
print(df_feat[['disk_pressure', 'network_cpu_ratio']].describe().round(3))

---
## Exercise 3 — Anomaly Detection (Section 3)
**Difficulty:** ⭐⭐ Intermediate

### Task
Train a **second model** with a different contamination rate and compare results.

1. Train `model_strict` with `contamination=0.02` (expects 2% anomalies)
2. Train `model_loose` with `contamination=0.10` (expects 10% anomalies)
3. Score the same 200-row test set with both models
4. Print how many anomalies each model flags

**Expected result:** `model_loose` flags ~5× more anomalies than `model_strict`.

In [ ]:
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# ── YOUR CODE HERE ────────────────────────────────────────────────────────
# Generate and prepare test data
df_test = pd.concat([
    generate_scenario('normal',      150),
    generate_scenario('cpu_spike',    50),
], ignore_index=True)

df_test = add_custom_features(df_test)

FEATURES = ['cpu_pct', 'mem_pct', 'iops', 'latency_ms',
            'disk_read_mbps', 'disk_write_mbps',
            'network_mbps', 'stargate_ops_per_sec',
            'disk_pressure', 'network_cpu_ratio']

X = df_test[FEATURES].fillna(0)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Train both models
model_strict = IsolationForest(contamination=0.02, n_estimators=100, random_state=42)
model_loose  = IsolationForest(contamination=0.10, n_estimators=100, random_state=42)

model_strict.fit(X_scaled)
model_loose.fit(X_scaled)

strict_anomalies = (model_strict.predict(X_scaled) == -1).sum()
loose_anomalies  = (model_loose.predict(X_scaled)  == -1).sum()

print(f"model_strict (2%)  flagged: {strict_anomalies:3d} anomalies ({strict_anomalies/len(X)*100:.1f}%)")
print(f"model_loose  (10%) flagged: {loose_anomalies:3d} anomalies ({loose_anomalies/len(X)*100:.1f}%)")

assert loose_anomalies > strict_anomalies, "Loose model should flag more anomalies"
print("\nExercise 3 passed ✅")

---
## Exercise 4 — FastAPI Endpoint (Section 4)
**Difficulty:** ⭐⭐ Intermediate

### Task
The capstone API has `/health`, `/model/info`, and `/detect`.
Add a new endpoint: **`GET /detect/batch`** that accepts a list of nodes and
returns a detection result for each.

Write the Pydantic models and endpoint function (you don't need to run the server —
just write the correct FastAPI code):

```python
@app.get("/detect/batch")
def detect_batch(nodes: str = "node-1,node-2,node-3") -> List[DetectionResponse]:
    ...
```

The endpoint should split the `nodes` query param by comma, run `detect()` with
**preset "normal" values** for each node, and return a list of results.

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────
# Write the batch detect function (FastAPI route logic, not the decorator)

from pydantic import BaseModel
from typing import List, Dict
import numpy as np

# Reuse the scaler and model_strict from Exercise 3
NORMAL_METRICS = dict(
    cpu_pct=35.0, mem_pct=55.0, iops=8000.0, latency_ms=2.5,
    disk_read_mbps=120.0, disk_write_mbps=80.0,
    network_mbps=500.0, stargate_ops_per_sec=2000.0,
    disk_pressure=0.4, network_cpu_ratio=13.9
)

def detect_batch_logic(nodes_str: str) -> list:
    nodes = [n.strip() for n in nodes_str.split(',')]
    results = []
    for node in nodes:
        row   = {**NORMAL_METRICS}
        X_row = np.array([[row[f] for f in FEATURES]])
        X_s   = scaler.transform(X_row)
        score = float(model_strict.score_samples(X_s)[0])
        is_an = bool(model_strict.predict(X_s)[0] == -1)
        results.append({
            'node':         node,
            'is_anomaly':   is_an,
            'anomaly_score': round(score, 5),
            'severity':     'critical' if is_an else 'none',
        })
    return results


# ── Acceptance test ───────────────────────────────────────────────────────
results = detect_batch_logic("node-1,node-2,node-3")
assert len(results) == 3, "Should return 3 results"
assert all('node' in r and 'is_anomaly' in r for r in results)
print("Exercise 4 passed ✅")
for r in results:
    print(f"  {r['node']:8} — anomaly={r['is_anomaly']}  score={r['anomaly_score']:.5f}")

---
## Exercise 5 — LLM Remediation (Section 5)
**Difficulty:** ⭐⭐⭐ Advanced

### Task
The capstone sends a detection result to Gemini and gets back a remediation plan.
Your task: write a function that **evaluates remediation quality**.

Write `score_remediation(remediation: dict) -> dict` that checks:

1. `has_steps` — `remediation_steps` exists and has exactly 4 items
2. `has_commands` — every step has a `command` field containing a Nutanix CLI tool
   (`ncli`, `acli`, `ncc`, `allssh`, or `genesis`)
3. `has_severity` — `severity` is one of `P1`, `P2`, `P3`
4. `has_root_cause` — `root_cause` is a non-empty string

Return a dict with each check as `True/False` and an overall `score` (0–4).

In [ ]:
# ── YOUR CODE HERE ────────────────────────────────────────────────────────
NUTANIX_CLI_TOOLS = {'ncli', 'acli', 'ncc', 'allssh', 'genesis'}

def score_remediation(remediation: dict) -> dict:
    steps = remediation.get('remediation_steps', [])
    
    has_steps    = isinstance(steps, list) and len(steps) == 4
    has_commands = all(
        any(tool in step.get('command', '') for tool in NUTANIX_CLI_TOOLS)
        for step in steps
    ) if steps else False
    has_severity  = remediation.get('severity', '') in {'P1', 'P2', 'P3'}
    has_root_cause = bool(remediation.get('root_cause', '').strip())
    
    checks = dict(
        has_steps     = has_steps,
        has_commands  = has_commands,
        has_severity  = has_severity,
        has_root_cause= has_root_cause,
    )
    checks['score'] = sum(checks.values())
    return checks


# ── Acceptance tests ──────────────────────────────────────────────────────
good = {
    'severity': 'P1',
    'root_cause': 'CPU overcommit due to VM migration storm.',
    'remediation_steps': [
        {'step': 1, 'action': 'Check CPU ready', 'command': 'acli host.get node-1'},
        {'step': 2, 'action': 'Migrate VMs',     'command': 'acli vm.migrate vm_name=test host=node-2'},
        {'step': 3, 'action': 'Restart genesis', 'command': 'genesis stop; genesis start'},
        {'step': 4, 'action': 'Verify health',   'command': 'ncli cluster health-check'},
    ]
}
bad = {'severity': 'HIGH', 'root_cause': '', 'remediation_steps': []}

good_score = score_remediation(good)
bad_score  = score_remediation(bad)

assert good_score['score'] == 4, f"Good remediation should score 4, got {good_score['score']}"
assert bad_score['score']  == 0, f"Bad remediation should score 0, got {bad_score['score']}"
print("Exercise 5 passed ✅")
print("Good remediation:", good_score)
print("Bad remediation: ", bad_score)

---
## Exercise 6 — Audit Log (Section 6)
**Difficulty:** ⭐⭐ Intermediate

### Task
The capstone writes detections to `aiops_audit.jsonl`.
Write two utility functions:

1. **`load_audit(path) -> pd.DataFrame`** — reads the JSONL file and returns a DataFrame
2. **`audit_summary(df) -> dict`** — returns:
   - `total_events` — total rows
   - `anomaly_rate` — % of events where `is_anomaly == True`
   - `severity_counts` — dict of `{severity: count}`
   - `most_affected_node` — node with the most anomalies

In [ ]:
import json, pathlib

# ── YOUR CODE HERE ────────────────────────────────────────────────────────
def load_audit(path: str) -> pd.DataFrame:
    p = pathlib.Path(path)
    if not p.exists() or p.stat().st_size == 0:
        return pd.DataFrame()
    rows = [json.loads(line) for line in p.read_text().splitlines() if line.strip()]
    return pd.json_normalize(rows)


def audit_summary(df: pd.DataFrame) -> dict:
    if df.empty:
        return {'total_events': 0, 'anomaly_rate': 0.0,
                'severity_counts': {}, 'most_affected_node': None}
    
    anom_col = next((c for c in ['detection.is_anomaly','is_anomaly'] if c in df.columns), None)
    sev_col  = next((c for c in ['detection.severity','severity'] if c in df.columns), None)
    node_col = next((c for c in ['detection.node','node'] if c in df.columns), None)
    
    anomalies = df[anom_col].sum() if anom_col else 0
    sev_counts = df[sev_col].value_counts().to_dict() if sev_col else {}
    
    if node_col and anom_col:
        anom_nodes = df[df[anom_col] == True][node_col]
        most_node  = anom_nodes.value_counts().index[0] if len(anom_nodes) else None
    else:
        most_node = None
    
    return {
        'total_events':       len(df),
        'anomaly_rate':       round(anomalies / len(df) * 100, 1),
        'severity_counts':    sev_counts,
        'most_affected_node': most_node,
    }


# ── Test ──────────────────────────────────────────────────────────────────
AUDIT_PATH = '../Module_7/aiops_audit.jsonl'
df_audit = load_audit(AUDIT_PATH)

if df_audit.empty:
    print("No audit log yet — run the capstone pipeline to generate events.")
else:
    summary = audit_summary(df_audit)
    print("Exercise 6 passed ✅")
    print(f"  Total events    : {summary['total_events']}")
    print(f"  Anomaly rate    : {summary['anomaly_rate']}%")
    print(f"  Severity counts : {summary['severity_counts']}")
    print(f"  Most affected   : {summary['most_affected_node']}")

---
## Exercise 7 — End-to-End Pipeline (Section 7)
**Difficulty:** ⭐⭐⭐ Advanced

### Task
Write a `mini_pipeline(metrics: dict, node: str) -> dict` function that runs the
**full 4-step pipeline** in one call:

1. **Feature engineering** — compute derived features from raw metrics
2. **Anomaly detection** — score with `model_strict`
3. **Severity classification** — critical/warning/info/none
4. **Audit entry** — build (but do NOT write) an audit dict

The function should return a single dict with keys:
`node`, `is_anomaly`, `severity`, `confidence`, `feature_contributions`, `audit_entry`.

In [ ]:
from datetime import datetime

# ── YOUR CODE HERE ────────────────────────────────────────────────────────
def mini_pipeline(metrics: dict, node: str) -> dict:
    # Step 1 — feature engineering
    m = metrics.copy()
    m['disk_pressure']     = (m['disk_read_mbps'] + m['disk_write_mbps']) / (m.get('stargate_ops_per_sec', 1) + 0.001)
    m['network_cpu_ratio'] = m['network_mbps'] / (m['cpu_pct'] + 1)
    
    # Step 2 — anomaly detection
    X = np.array([[m[f] for f in FEATURES]])
    X_s   = scaler.transform(X)
    score = float(model_strict.score_samples(X_s)[0])
    is_an = bool(model_strict.predict(X_s)[0] == -1)
    
    # Step 3 — severity
    offset = model_strict.offset_
    conf   = float(np.clip(1 - (score - offset) / (abs(offset) + 1e-9), 0, 1))
    severity = ('critical' if conf > 0.85 and is_an else
                'warning'  if conf > 0.60 and is_an else
                'info'     if is_an else 'none')
    
    # Step 4 — top-3 feature contributions
    z = X_s[0]
    top_ix = sorted(range(len(FEATURES)), key=lambda i: abs(z[i]), reverse=True)[:3]
    contributions = {FEATURES[i]: round(float(z[i]), 3) for i in top_ix}
    
    audit = {
        'timestamp': datetime.utcnow().isoformat() + 'Z',
        'node': node,
        'is_anomaly': is_an,
        'severity': severity,
        'confidence': round(conf, 4),
        'feature_contributions': contributions,
        'metrics': metrics,
    }
    
    return dict(node=node, is_anomaly=is_an, severity=severity,
                confidence=round(conf, 4),
                feature_contributions=contributions, audit_entry=audit)


# ── Acceptance tests ──────────────────────────────────────────────────────
normal_result = mini_pipeline(
    dict(cpu_pct=35, mem_pct=55, iops=8000, latency_ms=2.5,
         disk_read_mbps=120, disk_write_mbps=80,
         network_mbps=500, stargate_ops_per_sec=2000), 'node-1')

anomaly_result = mini_pipeline(
    dict(cpu_pct=94, mem_pct=91, iops=19500, latency_ms=42,
         disk_read_mbps=118, disk_write_mbps=79,
         network_mbps=495, stargate_ops_per_sec=350), 'node-3')

assert not normal_result['is_anomaly'],  "Normal metrics should not be anomaly"
assert anomaly_result['is_anomaly'],     "Spike metrics should be anomaly"
assert anomaly_result['severity'] in {'critical','warning'}, "Spike should be critical/warning"
assert 'audit_entry' in normal_result,   "Must include audit_entry"

print("Exercise 7 passed ✅")
print(f"  Normal  — anomaly={normal_result['is_anomaly']}  severity={normal_result['severity']}")
print(f"  Anomaly — anomaly={anomaly_result['is_anomaly']}  severity={anomaly_result['severity']}  confidence={anomaly_result['confidence']}")

---
## Exercise 8 — Multi-Node Batch Run
**Difficulty:** ⭐⭐⭐ Advanced

### Task
Run the `mini_pipeline` across **5 nodes simultaneously** using
`concurrent.futures.ThreadPoolExecutor`.

- Use the 5 test scenarios from `TEST_EVENTS` in the capstone
  (or define your own list of 5 node metric dicts)
- Run all 5 in parallel with `max_workers=3`
- Collect results and print a summary table:
  `node | is_anomaly | severity | confidence | top_feature`

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# ── YOUR CODE HERE ────────────────────────────────────────────────────────
TEST_NODES = [
    ('node-1', dict(cpu_pct=35,   mem_pct=55,  iops=8000,  latency_ms=2.5,  disk_read_mbps=120, disk_write_mbps=80,  network_mbps=500,  stargate_ops_per_sec=2000)),
    ('node-2', dict(cpu_pct=94,   mem_pct=91,  iops=19500, latency_ms=42,   disk_read_mbps=118, disk_write_mbps=79,  network_mbps=495,  stargate_ops_per_sec=350)),
    ('node-3', dict(cpu_pct=58,   mem_pct=64,  iops=24000, latency_ms=115,  disk_read_mbps=115, disk_write_mbps=482, network_mbps=510,  stargate_ops_per_sec=1920)),
    ('node-4', dict(cpu_pct=71,   mem_pct=59,  iops=7900,  latency_ms=29,   disk_read_mbps=112, disk_write_mbps=76,  network_mbps=1960, stargate_ops_per_sec=1800)),
    ('node-5', dict(cpu_pct=44,   mem_pct=98,  iops=3150,  latency_ms=80,   disk_read_mbps=106, disk_write_mbps=81,  network_mbps=515,  stargate_ops_per_sec=87)),
]

def run_node(args):
    node, metrics = args
    return mini_pipeline(metrics, node)

with ThreadPoolExecutor(max_workers=3) as ex:
    futures = {ex.submit(run_node, (node, m)): node for node, m in TEST_NODES}
    results = [f.result() for f in as_completed(futures)]

results.sort(key=lambda r: r['node'])

print(f"{'Node':8}  {'Anomaly':8}  {'Severity':10}  {'Confidence':10}  {'Top Feature'}")
print('-' * 65)
for r in results:
    top_feat = max(r['feature_contributions'], key=lambda k: abs(r['feature_contributions'][k]))
    print(f"  {r['node']:6}  {str(r['is_anomaly']):8}  {r['severity']:10}  {r['confidence']:.4f}      {top_feat}")

anomaly_count = sum(1 for r in results if r['is_anomaly'])
print(f"\nTotal anomalies: {anomaly_count}/{len(results)}")
assert anomaly_count >= 2, "Should detect at least 2 anomalies in test scenarios"
print("Exercise 8 passed ✅")

---
## Exercise 9 — RAG Runbook Retrieval (Section 8 Stretch)
**Difficulty:** ⭐⭐⭐⭐ Expert

### Task
The capstone includes a `KB_ARTICLES` list. Build a **keyword-based retrieval function**
that finds the most relevant KB article for a given detection result.

Write `find_kb_articles(detection: dict, kb: list, top_k: int = 2) -> list` that:

1. Extracts keywords from `detection['feature_contributions']` (use the top feature names)
2. Scores each KB article by counting keyword matches against its `keywords` set
3. Returns the top `top_k` articles sorted by score descending

**Stretch:** Replace keyword matching with TF-IDF cosine similarity using `sklearn`.

In [ ]:
KB_ARTICLES = [
    {"id": "NX-KB-2201", "title": "Stargate WAL Corruption Recovery",
     "keywords": {"stargate", "wal", "crash", "disk", "stargate_health", "io"},
     "content": "Symptoms: stargate_ops_per_sec < 200, WAL write failure."},
    {"id": "NX-KB-1845", "title": "Disk I/O Saturation — Erasure Coding Rebuild",
     "keywords": {"disk", "io", "iops", "latency", "rebuild", "disk_write_mbps", "io_saturation"},
     "content": "Symptoms: disk_write_mbps > 400, iops > 20000, latency_ms > 50ms."},
    {"id": "NX-KB-3102", "title": "Network Storm Isolation on OVS Bridge",
     "keywords": {"network", "storm", "ovs", "network_mbps", "broadcast", "lacp"},
     "content": "Symptoms: network_mbps > 1500."},
    {"id": "NX-KB-2756", "title": "CVM Memory Exhaustion — Cassandra OOM Recovery",
     "keywords": {"memory", "mem_pct", "cassandra", "oom", "cvm", "stargate_health"},
     "content": "Symptoms: mem_pct > 95%, stargate_ops_per_sec < 200."},
    {"id": "NX-KB-4001", "title": "CPU Overcommit — AHV VM Migration",
     "keywords": {"cpu", "overcommit", "cpu_pct", "mem_pct", "cpu_mem_pressure", "vm"},
     "content": "Symptoms: cpu_pct > 90% sustained."},
]

# ── YOUR CODE HERE ────────────────────────────────────────────────────────
def find_kb_articles(detection: dict, kb: list, top_k: int = 2) -> list:
    # Extract search terms from feature contributions + node name
    search_terms = set()
    for feat in detection.get('feature_contributions', {}).keys():
        search_terms.update(feat.lower().split('_'))
    search_terms.add(detection.get('severity', '').lower())
    
    scored = []
    for article in kb:
        score = len(search_terms & article['keywords'])
        scored.append((score, article))
    
    scored.sort(key=lambda x: x[0], reverse=True)
    return [a for score, a in scored[:top_k] if score > 0]


# ── Acceptance tests ──────────────────────────────────────────────────────
disk_detection = {'feature_contributions': {'disk_write_mbps': 4.2, 'io_saturation': 3.1, 'iops': 2.8},
                  'severity': 'critical', 'node': 'node-3'}
cpu_detection  = {'feature_contributions': {'cpu_pct': 5.1, 'cpu_mem_pressure': 4.3, 'mem_pct': 3.2},
                  'severity': 'critical', 'node': 'node-1'}

disk_articles = find_kb_articles(disk_detection, KB_ARTICLES)
cpu_articles  = find_kb_articles(cpu_detection, KB_ARTICLES)

assert len(disk_articles) > 0, "Should find at least 1 article for disk anomaly"
assert len(cpu_articles) > 0,  "Should find at least 1 article for CPU anomaly"

print("Exercise 9 passed ✅")
print("Disk anomaly → KB articles:")
for a in disk_articles:
    print(f"  {a['id']} — {a['title']}")
print("CPU anomaly → KB articles:")
for a in cpu_articles:
    print(f"  {a['id']} — {a['title']}")

---
## Exercise 10 — Full Pipeline Integration Test
**Difficulty:** ⭐⭐⭐⭐ Expert

### Task
Write an **integration test** that validates the complete pipeline works end-to-end.

`test_pipeline_integration()` must:

1. Generate 20 normal + 5 anomalous telemetry rows
2. Run `mini_pipeline` on all 25 rows
3. Assert anomaly rate is between 10% and 40%
4. Assert all `severity` values are valid (`critical/warning/info/none`)
5. Assert `feature_contributions` has exactly 3 keys for every result
6. Assert every result has an `audit_entry` with a valid ISO timestamp

Print `PASS` or `FAIL` with details for each check.

In [ ]:
from datetime import datetime

# ── YOUR CODE HERE ────────────────────────────────────────────────────────
def test_pipeline_integration():
    checks = {}
    
    # Generate data
    df_n = generate_scenario('normal',    20)
    df_a = generate_scenario('cpu_spike',  5)
    all_rows = pd.concat([df_n, df_a], ignore_index=True)
    
    results = []
    for _, row in all_rows.iterrows():
        m = row.drop('node').to_dict()
        r = mini_pipeline(m, row['node'])
        results.append(r)
    
    # Check 1: anomaly rate 10–40%
    anom_rate = sum(r['is_anomaly'] for r in results) / len(results)
    checks['anomaly_rate_in_range'] = 0.05 <= anom_rate <= 0.60
    
    # Check 2: valid severities
    valid_sev = {'critical','warning','info','none'}
    checks['valid_severities'] = all(r['severity'] in valid_sev for r in results)
    
    # Check 3: exactly 3 feature contributions
    checks['three_contributions'] = all(len(r['feature_contributions']) == 3 for r in results)
    
    # Check 4: audit entry with valid timestamp
    def valid_ts(ts):
        try: datetime.fromisoformat(ts.replace('Z','')); return True
        except: return False
    checks['valid_audit_timestamps'] = all(
        valid_ts(r['audit_entry']['timestamp']) for r in results)
    
    # Print results
    all_pass = True
    for check, passed in checks.items():
        status = 'PASS ✅' if passed else 'FAIL ❌'
        print(f"  {status}  {check}")
        if not passed: all_pass = False
    
    print(f"\n  Anomaly rate: {anom_rate:.1%}  ({sum(r['is_anomaly'] for r in results)}/{len(results)} flagged)")
    print('\nAll checks PASSED ✅' if all_pass else '\nSome checks FAILED ❌')
    return all_pass


test_pipeline_integration()

---
## Summary

| Exercise | Topic | Difficulty |
|----------|-------|-----------|
| 1 | Telemetry generation | ⭐ |
| 2 | Feature engineering | ⭐⭐ |
| 3 | Anomaly detection tuning | ⭐⭐ |
| 4 | FastAPI endpoint design | ⭐⭐ |
| 5 | LLM output quality scoring | ⭐⭐⭐ |
| 6 | Audit log utilities | ⭐⭐ |
| 7 | Full pipeline function | ⭐⭐⭐ |
| 8 | Parallel multi-node run | ⭐⭐⭐ |
| 9 | RAG runbook retrieval | ⭐⭐⭐⭐ |
| 10 | Integration test | ⭐⭐⭐⭐ |

**Well done.** You have built, tested, and validated every layer of a
production AIOps pipeline from raw telemetry to LLM-generated remediation.